# Capture Live Segmentation Adaptation Frames

Capture a better set of webcam frames specifically for live segmentation improvement. This notebook is stricter than the live try-on notebook: it is for collecting trainable webcam-like frames, not for rendering hairstyles.

Recommended capture conditions:
- near-frontal face
- hair fully visible
- steady camera distance
- avoid extreme backlight if possible
- collect small pose variation, but stay within the supported range

In [1]:
from pathlib import Path
import sys

import cv2
import pandas as pd
from PIL import Image
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from app.core.face_analyzer import analyze_face_pil_image
from app.core.hair_segmentation import predict_hair_mask_image_from_face_roi
from app.core.live_support import evaluate_supported_live_range, summarize_live_mask_quality

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
CAMERA_INDEX = 0
MAX_FRAMES = 240
DISPLAY_WIDTH = 1280
SAVE_EVERY_N_FRAMES = 10
MAX_SAVED_FRAMES = 30
REQUIRE_SUPPORTED_FRAME = True
REQUIRE_NON_EMPTY_ROI_MASK = True

OUTPUT_DIR = PROJECT_ROOT / 'backend' / 'outputs' / 'live_segmentation_eval' / 'inputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output dir:', OUTPUT_DIR)

Output dir: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\outputs\live_segmentation_eval\inputs


In [3]:
capture = cv2.VideoCapture(CAMERA_INDEX)
if not capture.isOpened():
    raise RuntimeError(f'Could not open webcam index {CAMERA_INDEX}.')

frame_index = 0
saved_count = 0
rows = []
print('Starting capture... press q to stop early.')

try:
    while frame_index < MAX_FRAMES and saved_count < MAX_SAVED_FRAMES:
        ok, frame = capture.read()
        if not ok:
            break

        frame_index += 1
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(rgb)
        analysis = analyze_face_pil_image(pil_image)
        support = evaluate_supported_live_range(pil_image, analysis)

        roi_mask = None
        roi_quality = {
            'mask_quality_reason': 'no_face_bbox',
            'mask_reliable': False,
            'mask_nonzero_ratio': 0.0,
        }
        if analysis.face_bbox is not None:
            roi_mask = predict_hair_mask_image_from_face_roi(pil_image, analysis.face_bbox)
            roi_quality = summarize_live_mask_quality(analysis, roi_mask)

        allow_save = frame_index % SAVE_EVERY_N_FRAMES == 0
        if REQUIRE_SUPPORTED_FRAME and not support['frame_supported']:
            allow_save = False
        if REQUIRE_NON_EMPTY_ROI_MASK and roi_quality['mask_nonzero_ratio'] <= 0.0:
            allow_save = False

        status = f"support={support['support_reason']} | roi={roi_quality['mask_quality_reason']} | saved={saved_count}"
        preview = frame.copy()
        cv2.putText(preview, status, (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

        display_frame = preview
        if DISPLAY_WIDTH and preview.shape[1] > DISPLAY_WIDTH:
            scale = DISPLAY_WIDTH / preview.shape[1]
            display_frame = cv2.resize(preview, None, fx=scale, fy=scale)

        cv2.imshow('Live Segmentation Capture', display_frame)

        if allow_save:
            saved_count += 1
            image_path = OUTPUT_DIR / f'live_frame_{saved_count:04d}.png'
            pil_image.save(image_path)
            rows.append({
                'filename': image_path.name,
                'support_reason': support['support_reason'],
                'frame_supported': bool(support['frame_supported']),
                'roi_mask_reason': roi_quality['mask_quality_reason'],
                'roi_mask_reliable': bool(roi_quality['mask_reliable']),
                'roi_mask_coverage': float(roi_quality['mask_nonzero_ratio']),
            })

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
finally:
    capture.release()
    cv2.destroyAllWindows()

print('Processed frames:', frame_index)
print('Saved frames:', saved_count)
summary_df = pd.DataFrame(rows)
display(summary_df.head())

Starting capture... press q to stop early.
Processed frames: 240
Saved frames: 22


,filename,support_reason,frame_supported,roi_mask_reason,roi_mask_reliable,roi_mask_coverage
0,live_frame_0001.png,supported,True,mask_too_narrow,False,0.007780
1,live_frame_0002.png,supported,True,reliable,True,0.015023
2,live_frame_0003.png,supported,True,reliable,True,0.012396
3,live_frame_0004.png,supported,True,mask_too_narrow,False,0.000010
4,live_frame_0005.png,supported,True,mask_too_narrow,False,0.005521


In [4]:
if not summary_df.empty:
    print(summary_df['support_reason'].value_counts())
    print(summary_df['roi_mask_reason'].value_counts())

support_reason
supported    22
Name: count, dtype: int64
roi_mask_reason
mask_too_narrow    20
reliable            2
Name: count, dtype: int64
